# 2d_HM_grouped_distribution_7b

Grouped pooled-distribution summaries for the matched 7/8B model set, pooled across all three control variants. Each row is a model; within each operation/entity slice we compare the model's pooled answer distribution against the human pooled distribution, then average those slice-level metrics. Both inclusive and abstention-filtered versions are exported.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

_NOTEBOOK_DIR = Path.cwd()
if (_NOTEBOOK_DIR / "helpers.py").exists():
    sys.path.insert(0, str(_NOTEBOOK_DIR.parent))
elif (_NOTEBOOK_DIR / "notebooks" / "helpers.py").exists():
    sys.path.insert(0, str(_NOTEBOOK_DIR))

from notebooks.helpers import (
    ROOT,
    LATEX_TABLES,
    pretty_print_path,
    hh_question_means,
    hm_question_means,
    variant_summary_table,
    pairwise_variant_correlation_table,
    grouped_pattern_table,
    scatter_correlation_table,
    blind_accuracy_summary,
    qualitative_qdf,
    attach_answer_summaries,
    hh_ranked_examples,
    variant_top_bottom_table,
    hh_degradation_table,
    top_questions_tables,
    to_latex_table,
)

from notebooks.helpers import grouped_distribution_long_table, grouped_distribution_wide_table


In [ ]:
op_wide = grouped_distribution_wide_table(
    "op",
    condition="inst_blind",
    variants=("C", "B", "A"),
    filter_abstention=False,
)
op_wide_filt = grouped_distribution_wide_table(
    "op",
    condition="inst_blind",
    variants=("C", "B", "A"),
    filter_abstention=True,
)
ent_wide = grouped_distribution_wide_table(
    "ent",
    condition="inst_blind",
    variants=("C", "B", "A"),
    filter_abstention=False,
)
ent_wide_filt = grouped_distribution_wide_table(
    "ent",
    condition="inst_blind",
    variants=("C", "B", "A"),
    filter_abstention=True,
)

display(op_wide)
display(op_wide_filt)
display(ent_wide)
display(ent_wide_filt)

In [ ]:
out = LATEX_TABLES / "hm_grouped_distribution_7b_op_all_variants.tex"
to_latex_table(
    op_wide,
    out,
    "Matched 7/8B operation-group pooled-distribution alignment to humans across all control variants (inclusive version). Lower JS/TV indicate closer alignment; Cramer's V summarizes residual distribution mismatch; abstention rate is measured before any filtering.",
    "tab:hm_grouped_distribution_7b_op",
    float_formatters={col: ".3f" for col in op_wide.columns if "|" in col},
)
print(pretty_print_path(out))

In [ ]:
out = LATEX_TABLES / "hm_grouped_distribution_7b_op_all_variants_filtered.tex"
to_latex_table(
    op_wide_filt,
    out,
    "Matched 7/8B operation-group pooled-distribution alignment to humans across all control variants (shared substantive-response version). Lower JS/TV indicate closer alignment after removing abstentions and comparing on the shared non-abstaining subset.",
    "tab:hm_grouped_distribution_7b_op_filtered",
    float_formatters={col: ".3f" for col in op_wide_filt.columns if "|" in col},
)
print(pretty_print_path(out))

In [ ]:
out = LATEX_TABLES / "hm_grouped_distribution_7b_ent_all_variants.tex"
to_latex_table(
    ent_wide,
    out,
    "Matched 7/8B entity-group pooled-distribution alignment to humans across all control variants (inclusive version). Lower JS/TV indicate closer alignment; Cramer's V summarizes residual distribution mismatch; abstention rate is measured before any filtering.",
    "tab:hm_grouped_distribution_7b_ent",
    float_formatters={col: ".3f" for col in ent_wide.columns if "|" in col},
)
print(pretty_print_path(out))

In [ ]:
out = LATEX_TABLES / "hm_grouped_distribution_7b_ent_all_variants_filtered.tex"
to_latex_table(
    ent_wide_filt,
    out,
    "Matched 7/8B entity-group pooled-distribution alignment to humans across all control variants (shared substantive-response version). Lower JS/TV indicate closer alignment after removing abstentions and comparing on the shared non-abstaining subset.",
    "tab:hm_grouped_distribution_7b_ent_filtered",
    float_formatters={col: ".3f" for col in ent_wide_filt.columns if "|" in col},
)
print(pretty_print_path(out))

## Relation to HH--HM Scatter

The HH--HM scatterplots ask whether a model preserves the *question-level* human agreement structure. The grouped pooled-distribution tables ask a different but complementary question: within broad operation/entity slices, does the model place answer mass on the same categories that humans do? A model can be close to the HH--HM diagonal yet still show a non-human pooled distribution within count or yes/no slices, so the two views should be read together.

In [ ]:
for label, table in [
    ("Operation, inclusive", op_wide),
    ("Operation, filtered", op_wide_filt),
    ("Entity, inclusive", ent_wide),
    ("Entity, filtered", ent_wide_filt),
]:
    print(label)
    winners = table.iloc[0][["Model", "Group"] + [c for c in table.columns if c.endswith("Mean JS")][:3]]
    display(winners.to_frame().T)